In [6]:
import os
import requests
import pandas as pd
from datetime import date

# ==== CONFIG ====
API_KEY = '766adf50b56b3ba5b59275e345016de1'
SPORT = 'baseball_mlb'
REGIONS = 'us'
MARKETS = 'h2h,spreads'
ODDS_FORMAT = 'american'
DATE_FORMAT = 'iso'

today = date.today().isoformat()
folder = 'daily_odds'
filename = f'daily_odds_{today}.csv'
output_path = os.path.join(folder, filename)

# ==== Fetch Odds ====
odds_response = requests.get(
    f'https://api.the-odds-api.com/v4/sports/{SPORT}/odds',
    params={
        'api_key': API_KEY,
        'regions': REGIONS,
        'markets': MARKETS,
        'oddsFormat': ODDS_FORMAT,
        'dateFormat': DATE_FORMAT,
    }
)

if odds_response.status_code != 200:
    print(f'❌ Failed to get odds: status_code {odds_response.status_code}, response body {odds_response.text}')
else:
    odds_json = odds_response.json()
    print(f'✅ Number of MLB events: {len(odds_json)}')

    rows = []
    for game in odds_json:
        home_team = game.get('home_team')
        away_team = game.get('away_team')
        commence_time = game.get('commence_time')

        moneyline = {}
        spread = {}

        for book in game.get('bookmakers', []):
            bookmaker = book['title']

            for market in book.get('markets', []):
                if market['key'] == 'h2h':
                    for outcome in market.get('outcomes', []):
                        moneyline[outcome['name']] = outcome['price']

                if market['key'] == 'spreads':
                    outcomes = market.get('outcomes', [])
                    if len(outcomes) == 2:
                        team1, team2 = outcomes
                        if team1['point'] < team2['point']:
                            spread = {
                                'Favorite': team1['name'],
                                'Underdog': team2['name'],
                                'Spread': team1['point']
                            }
                        else:
                            spread = {
                                'Favorite': team2['name'],
                                'Underdog': team1['name'],
                                'Spread': team2['point']
                            }

            if moneyline and spread:
                rows.append({
                    'Commence_Time': commence_time,
                    'Home_Team': home_team,
                    'Away_Team': away_team,
                    'Bookmaker': bookmaker,
                    'Favorite': spread.get('Favorite'),
                    'Underdog': spread.get('Underdog'),
                    'Spread': spread.get('Spread'),
                    'Home_ML': moneyline.get(home_team),
                    'Away_ML': moneyline.get(away_team)
                })
                break  # Stop at first bookmaker with both markets

    df = pd.DataFrame(rows)

    # Save
    os.makedirs(folder, exist_ok=True)
    df.to_csv(output_path, index=False)

    print(f"📁 Saved combined moneyline + spread odds to {output_path}")
    print(df.head())
    print("🔄 API Quota Remaining:", odds_response.headers.get('x-requests-remaining'))
    print("📊 API Requests Used:", odds_response.headers.get('x-requests-used'))


✅ Number of MLB events: 20
📁 Saved combined moneyline + spread odds to daily_odds/daily_odds_2025-05-26.csv
          Commence_Time             Home_Team            Away_Team  \
0  2025-05-26T22:10:00Z   Cleveland Guardians  Los Angeles Dodgers   
1  2025-05-26T23:06:00Z        Tampa Bay Rays      Minnesota Twins   
2  2025-05-27T00:11:00Z  Arizona Diamondbacks   Pittsburgh Pirates   
3  2025-05-27T00:41:00Z      San Diego Padres        Miami Marlins   
4  2025-05-27T01:38:00Z    Los Angeles Angels     New York Yankees   

      Bookmaker              Favorite             Underdog  Spread  Home_ML  \
0       FanDuel   Los Angeles Dodgers  Cleveland Guardians    -5.5     2000   
1       FanDuel        Tampa Bay Rays      Minnesota Twins    -1.5     -154   
2       FanDuel  Arizona Diamondbacks   Pittsburgh Pirates    -1.5     -190   
3       FanDuel      San Diego Padres        Miami Marlins    -1.5     -152   
4  BetOnline.ag      New York Yankees   Los Angeles Angels    -1.5      161 